In [1]:
import os
import time
import numpy as np
import scanpy as sc

INPUT_DIR = "/old_nfs/chengwang_data/ICML_data/sim_data/globe"
OUTPUT_ROOT = "/old_nfs/chengwang_data/ICML_data/benchmark_results/pca"

DEPTHS = [10000, 5000, 2000, 200]

N_COMPONENTS = 16
USE_HVG = True
N_TOP_GENES = 2000
SEED = 0

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("OUTPUT_ROOT =", OUTPUT_ROOT)
def run_pca_one_depth(depth,
                      input_dir=INPUT_DIR,
                      output_root=OUTPUT_ROOT,
                      n_components=N_COMPONENTS,
                      use_hvg=USE_HVG,
                      n_top_genes=N_TOP_GENES,
                      seed=SEED):
    infile = os.path.join(input_dir, f"adata_depth_{depth}.h5ad")
    outdir = os.path.join(output_root, f"depth_{depth}")
    os.makedirs(outdir, exist_ok=True)

    print(f"\n===== PCA | depth={depth} =====")
    print("reading:", infile)

    adata = sc.read_h5ad(infile)
    adata_out = adata.copy()

    # PCA 输入对象
    adata_pca = adata.copy()

    # 是否先做 HVG
    if use_hvg:
        sc.pp.normalize_total(adata_pca, target_sum=1e4)
        sc.pp.log1p(adata_pca)
        sc.pp.highly_variable_genes(
            adata_pca,
            n_top_genes=n_top_genes,
            flavor="seurat"
        )
        adata_pca = adata_pca[:, adata_pca.var["highly_variable"]].copy()
        print("after HVG:", adata_pca.shape)
    else:
        # 如果不用 HVG，也建议做标准 PCA 前处理
        sc.pp.normalize_total(adata_pca, target_sum=1e4)
        sc.pp.log1p(adata_pca)
        print("using all genes:", adata_pca.shape)

    t0 = time.time()

    sc.pp.pca(
        adata_pca,
        n_comps=n_components,
        svd_solver="arpack",
        random_state=seed
    )

    t1 = time.time()
    runtime_seconds = t1 - t0

    emb = adata_pca.obsm["X_pca"].copy()

    print("embedding shape:", emb.shape)
    print(f"runtime_seconds: {runtime_seconds:.2f}")

    # 写回原始 adata，保持细胞顺序一致
    adata_out.obsm["X_pca"] = emb
    adata_out.uns["pca_runtime_seconds"] = float(runtime_seconds)
    adata_out.uns["pca_runtime_minutes"] = float(runtime_seconds / 60)
    adata_out.uns["pca_depth"] = int(depth)
    adata_out.uns["pca_n_components"] = int(n_components)
    adata_out.uns["pca_use_hvg"] = bool(use_hvg)
    adata_out.uns["pca_n_top_genes"] = int(n_top_genes) if use_hvg else None

    np.save(os.path.join(outdir, "X_pca.npy"), emb)
    adata_out.write(os.path.join(outdir, f"adata_depth_{depth}_pca.h5ad"))

    with open(os.path.join(outdir, "runtime.txt"), "w") as f:
        f.write(f"runtime_seconds\t{runtime_seconds:.6f}\n")
        f.write(f"runtime_minutes\t{runtime_seconds/60:.6f}\n")

    print("saved to:", outdir)
    

OUTPUT_ROOT = /old_nfs/chengwang_data/ICML_data/benchmark_results/pca


/home/chongxiao/miniconda3/envs/htodemux/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
for depth in DEPTHS:
    run_pca_one_depth(depth)

print("\nAll PCA jobs finished.")
for depth in DEPTHS:
    f = os.path.join(OUTPUT_ROOT, f"depth_{depth}", f"adata_depth_{depth}_pca.h5ad")
    ad = sc.read_h5ad(f)
    print(depth, ad.obsm["X_pca"].shape)


===== PCA | depth=10000 =====
reading: /old_nfs/chengwang_data/ICML_data/sim_data/globe/adata_depth_10000.h5ad


/home/chongxiao/miniconda3/envs/htodemux/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


after HVG: (240, 300)
embedding shape: (240, 16)
runtime_seconds: 0.19
saved to: /old_nfs/chengwang_data/ICML_data/benchmark_results/pca/depth_10000

===== PCA | depth=5000 =====
reading: /old_nfs/chengwang_data/ICML_data/sim_data/globe/adata_depth_5000.h5ad
after HVG: (240, 300)
embedding shape: (240, 16)
runtime_seconds: 0.01
saved to: /old_nfs/chengwang_data/ICML_data/benchmark_results/pca/depth_5000

===== PCA | depth=2000 =====
reading: /old_nfs/chengwang_data/ICML_data/sim_data/globe/adata_depth_2000.h5ad
after HVG: (240, 300)
embedding shape: (240, 16)
runtime_seconds: 0.01
saved to: /old_nfs/chengwang_data/ICML_data/benchmark_results/pca/depth_2000

===== PCA | depth=200 =====
reading: /old_nfs/chengwang_data/ICML_data/sim_data/globe/adata_depth_200.h5ad
after HVG: (240, 294)
embedding shape: (240, 16)
runtime_seconds: 0.01
saved to: /old_nfs/chengwang_data/ICML_data/benchmark_results/pca/depth_200

All PCA jobs finished.
10000 (240, 16)
5000 (240, 16)
2000 (240, 16)
200 (240, 

/home/chongxiao/miniconda3/envs/htodemux/lib/python3.10/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: `n_top_genes` > number of normalized dispersions, returning all genes with normalized dispersions.
  return fn(*args_all, **kw)


In [20]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
GT_PATH = "/old_nfs/chengwang_data/ICML_data/sim_data/low_dim_sphere.h5ad"

METHOD_CONFIG = {
    "scvi": {
        "base_dir": "/old_nfs/chengwang_data/ICML_data/benchmark_results/scvi",
        "file_pattern": "adata_depth_{depth}_scvi.h5ad",
        "obsm_key": "X_scVI",
        "metric": "euclidean",
    },
    "scphere": {
        "base_dir": "/old_nfs/chengwang_data/ICML_data/benchmark_results/scphere_multi_seed",
        "file_pattern": "adata_depth_{depth}_scphere.h5ad",
        "obsm_key": "X_scPhere",
        "metric": "arc",
    },
    "irvae": {
        "base_dir": "/old_nfs/chengwang_data/ICML_data/benchmark_results/irvae",
        "file_pattern": "adata_depth_{depth}_irvae.h5ad",
        "obsm_key": "X_irvae",
        "metric": "euclidean",
    },
    "flatvi": {
        "base_dir": "/old_nfs/chengwang_data/ICML_data/benchmark_results/flatvi",
        "file_pattern": "adata_depth_{depth}_flatvi.h5ad",
        "obsm_key": "X_flatvi",
        "metric": "euclidean",
    },
    "pca": {
        "base_dir": "/old_nfs/chengwang_data/ICML_data/benchmark_results/pca",
        "file_pattern": "adata_depth_{depth}_pca.h5ad",
        "obsm_key": "X_pca",
        "metric": "euclidean",
    },
}

DEPTHS = [10000, 5000, 2000, 200]

OUT_CSV = "/old_nfs/chengwang_data/ICML_data/benchmark_results/embedding_distance_spearman.csv"

In [21]:
def upper_triangle_euclidean(X):
    """
    返回所有细胞对的欧氏距离上三角向量
    """
    X = np.asarray(X, dtype=np.float64)
    return pdist(X, metric="euclidean")


def upper_triangle_arc(X, eps=1e-12):
    """
    返回所有细胞对的弧距离上三角向量
    d(x,y) = arccos( <x,y> / (||x|| ||y||) )
    """
    X = np.asarray(X, dtype=np.float64)

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.clip(norms, eps, None)
    Xn = X / norms

    # 先算 cosine similarity，再转弧距离
    cos_vec = pdist(Xn, metric="cosine")   # cosine distance = 1 - cosine similarity
    sim_vec = 1.0 - cos_vec
    sim_vec = np.clip(sim_vec, -1.0, 1.0)

    arc_vec = np.arccos(sim_vec)
    return arc_vec


def compute_distance_vector(X, metric):
    if metric == "euclidean":
        return upper_triangle_euclidean(X)
    elif metric == "arc":
        return upper_triangle_arc(X)
    else:
        raise ValueError(f"Unsupported metric: {metric}")

In [22]:
gt_adata = sc.read_h5ad(GT_PATH)
print("GT shape:", gt_adata.shape)
def load_method_embedding(method_name, depth):
    cfg = METHOD_CONFIG[method_name]
    f = os.path.join(
        cfg["base_dir"],
        f"depth_{depth}",
        cfg["file_pattern"].format(depth=depth)
    )
    adata = sc.read_h5ad(f)
    X = adata.obsm[cfg["obsm_key"]]
    return adata, X


def align_with_ground_truth(method_adata, method_X, gt_adata):
    """
    按 obs_names 对齐 method 和 ground truth
    """
    common = method_adata.obs_names.intersection(gt_adata.obs_names)
    if len(common) == 0:
        raise ValueError("No overlapping cells between method adata and ground truth.")

    method_idx = method_adata.obs_names.get_indexer(common)
    gt_idx = gt_adata.obs_names.get_indexer(common)

    method_X_aligned = np.asarray(method_X)[method_idx]
    gt_X_aligned = np.asarray(gt_adata.X)[gt_idx]

    return common, method_X_aligned, gt_X_aligned

GT shape: (240, 22)


In [23]:
def evaluate_one(method_name, depth, gt_adata):
    cfg = METHOD_CONFIG[method_name]
    metric = cfg["metric"]

    method_adata, method_X = load_method_embedding(method_name, depth)
    common_cells, method_X_aligned, gt_X_aligned = align_with_ground_truth(
        method_adata, method_X, gt_adata
    )

    print(f"{method_name} | depth={depth} | n_cells={len(common_cells)} | metric={metric}")

    dist_method = compute_distance_vector(method_X_aligned, metric=metric)
    dist_gt = compute_distance_vector(gt_X_aligned, metric=metric)

    corr, pval = spearmanr(dist_method, dist_gt)

    return {
        "method": method_name,
        "depth": depth,
        "metric": metric,
        "n_cells": len(common_cells),
        "n_pairs": len(dist_method),
        "spearman": corr,
        "pvalue": pval,
    }
results = []

for method_name in METHOD_CONFIG:
    for depth in DEPTHS:
        try:
            res = evaluate_one(method_name, depth, gt_adata)
            results.append(res)
        except Exception as e:
            print(f"FAILED: method={method_name}, depth={depth}, error={e}")
            results.append({
                "method": method_name,
                "depth": depth,
                "metric": METHOD_CONFIG[method_name]["metric"],
                "n_cells": np.nan,
                "n_pairs": np.nan,
                "spearman": np.nan,
                "pvalue": np.nan,
            })

results_df = pd.DataFrame(results)
results_df

scvi | depth=10000 | n_cells=240 | metric=euclidean
scvi | depth=5000 | n_cells=240 | metric=euclidean
scvi | depth=2000 | n_cells=240 | metric=euclidean
scvi | depth=200 | n_cells=240 | metric=euclidean
FAILED: method=scphere, depth=10000, error=[Errno 2] Unable to synchronously open file (unable to open file: name = '/old_nfs/chengwang_data/ICML_data/benchmark_results/scphere_multi_seed/depth_10000/adata_depth_10000_scphere.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)
FAILED: method=scphere, depth=5000, error=[Errno 2] Unable to synchronously open file (unable to open file: name = '/old_nfs/chengwang_data/ICML_data/benchmark_results/scphere_multi_seed/depth_5000/adata_depth_5000_scphere.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)
FAILED: method=scphere, depth=2000, error=[Errno 2] Unable to synchronously open file (unable to open file: name = '/old_nfs/chengwang_data/ICML_data/benchmark_results/scphe

,method,depth,metric,n_cells,n_pairs,spearman,pvalue
0,scvi,10000,euclidean,240.0,28680.0,0.294170,0.0
1,scvi,5000,euclidean,240.0,28680.0,0.289472,0.0
2,scvi,2000,euclidean,240.0,28680.0,0.327126,0.0
3,scvi,200,euclidean,240.0,28680.0,0.237171,0.0
4,scphere,10000,arc,NaN,NaN,NaN,NaN
5,scphere,5000,arc,NaN,NaN,NaN,NaN
6,scphere,2000,arc,NaN,NaN,NaN,NaN
7,scphere,200,arc,NaN,NaN,NaN,NaN
8,irvae,10000,euclidean,240.0,28680.0,0.347661,0.0
9,irvae,5000,euclidean,240.0,28680.0,0.356604,0.0


In [25]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

GT_PATH = "/old_nfs/chengwang_data/ICML_data/sim_data/low_dim_sphere.h5ad"
SCPHERE_DIR = "/old_nfs/chengwang_data/ICML_data/benchmark_results/scphere_multi_seed"

DEPTHS = [10000, 5000, 2000, 200]
SEEDS = [0, 1, 2, 3, 4]

def upper_triangle_arc(X, eps=1e-12):
    """
    弧距离:
    d(x,y) = arccos( <x,y> / (||x|| ||y||) )
    返回所有细胞对的上三角距离向量
    """
    X = np.asarray(X, dtype=np.float64)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.clip(norms, eps, None)
    Xn = X / norms

    cos_dist = pdist(Xn, metric="cosine")
    cos_sim = 1.0 - cos_dist
    cos_sim = np.clip(cos_sim, -1.0, 1.0)

    arc_dist = np.arccos(cos_sim)
    return arc_dist

gt_adata = sc.read_h5ad(GT_PATH)

results = []

for depth in DEPTHS:
    for seed in SEEDS:
        scphere_path = os.path.join(
            SCPHERE_DIR,
            f"depth_{depth}",
            f"seed_{seed}",
            f"adata_depth_{depth}_scphere_seed_{seed}.h5ad"
        )

        print(f"\n===== scPhere | depth={depth} | seed={seed} =====")
        print("reading:", scphere_path)

        if not os.path.exists(scphere_path):
            print("file not found")
            results.append({
                "method": "scphere",
                "depth": depth,
                "seed": seed,
                "n_cells": np.nan,
                "n_pairs": np.nan,
                "spearman": np.nan,
                "pvalue": np.nan,
                "reason": "file_not_found",
            })
            continue

        adata = sc.read_h5ad(scphere_path)
        emb = adata.obsm["X_scPhere"]

        common_cells = adata.obs_names.intersection(gt_adata.obs_names)
        print("n_common_cells:", len(common_cells))

        if len(common_cells) < 3:
            results.append({
                "method": "scphere",
                "depth": depth,
                "seed": seed,
                "n_cells": len(common_cells),
                "n_pairs": np.nan,
                "spearman": np.nan,
                "pvalue": np.nan,
                "reason": "too_few_common_cells",
            })
            continue

        idx_scphere = adata.obs_names.get_indexer(common_cells)
        idx_gt = gt_adata.obs_names.get_indexer(common_cells)

        emb_aligned = np.asarray(emb)[idx_scphere]
        gt_aligned = np.asarray(gt_adata.X)[idx_gt]

        if np.isnan(emb_aligned).any() or np.isinf(emb_aligned).any():
            results.append({
                "method": "scphere",
                "depth": depth,
                "seed": seed,
                "n_cells": len(common_cells),
                "n_pairs": np.nan,
                "spearman": np.nan,
                "pvalue": np.nan,
                "reason": "embedding_has_nan_or_inf",
            })
            continue

        if np.isnan(gt_aligned).any() or np.isinf(gt_aligned).any():
            results.append({
                "method": "scphere",
                "depth": depth,
                "seed": seed,
                "n_cells": len(common_cells),
                "n_pairs": np.nan,
                "spearman": np.nan,
                "pvalue": np.nan,
                "reason": "gt_has_nan_or_inf",
            })
            continue

        dist_scphere = upper_triangle_arc(emb_aligned)
        dist_gt = upper_triangle_arc(gt_aligned)

        n_unique_scphere = len(np.unique(np.round(dist_scphere, 10)))
        n_unique_gt = len(np.unique(np.round(dist_gt, 10)))

        print("unique dist_scphere:", n_unique_scphere)
        print("unique dist_gt:", n_unique_gt)

        if n_unique_scphere <= 1:
            results.append({
                "method": "scphere",
                "depth": depth,
                "seed": seed,
                "n_cells": len(common_cells),
                "n_pairs": len(dist_scphere),
                "spearman": np.nan,
                "pvalue": np.nan,
                "reason": "constant_scphere_distance",
            })
            continue

        if n_unique_gt <= 1:
            results.append({
                "method": "scphere",
                "depth": depth,
                "seed": seed,
                "n_cells": len(common_cells),
                "n_pairs": len(dist_gt),
                "spearman": np.nan,
                "pvalue": np.nan,
                "reason": "constant_gt_distance",
            })
            continue

        corr, pval = spearmanr(dist_scphere, dist_gt)

        results.append({
            "method": "scphere",
            "depth": depth,
            "seed": seed,
            "n_cells": len(common_cells),
            "n_pairs": len(dist_scphere),
            "spearman": corr,
            "pvalue": pval,
            "reason": "ok",
        })

results_df = pd.DataFrame(results).sort_values(
    ["depth", "seed"], ascending=[False, True]
).reset_index(drop=True)

results_df


===== scPhere | depth=10000 | seed=0 =====
reading: /old_nfs/chengwang_data/ICML_data/benchmark_results/scphere_multi_seed/depth_10000/seed_0/adata_depth_10000_scphere_seed_0.h5ad
n_common_cells: 240
unique dist_scphere: 28678
unique dist_gt: 28680

===== scPhere | depth=10000 | seed=1 =====
reading: /old_nfs/chengwang_data/ICML_data/benchmark_results/scphere_multi_seed/depth_10000/seed_1/adata_depth_10000_scphere_seed_1.h5ad
n_common_cells: 240
unique dist_scphere: 28680
unique dist_gt: 28680

===== scPhere | depth=10000 | seed=2 =====
reading: /old_nfs/chengwang_data/ICML_data/benchmark_results/scphere_multi_seed/depth_10000/seed_2/adata_depth_10000_scphere_seed_2.h5ad
n_common_cells: 240
unique dist_scphere: 28680
unique dist_gt: 28680

===== scPhere | depth=10000 | seed=3 =====
reading: /old_nfs/chengwang_data/ICML_data/benchmark_results/scphere_multi_seed/depth_10000/seed_3/adata_depth_10000_scphere_seed_3.h5ad
n_common_cells: 240
unique dist_scphere: 28676
unique dist_gt: 28680


,method,depth,seed,n_cells,n_pairs,spearman,pvalue,reason
0,scphere,10000,0,240,28680,0.590871,0.000000e+00,ok
1,scphere,10000,1,240,28680,0.640023,0.000000e+00,ok
2,scphere,10000,2,240,28680,0.689824,0.000000e+00,ok
3,scphere,10000,3,240,28680,0.599247,0.000000e+00,ok
4,scphere,10000,4,240,28680,0.663835,0.000000e+00,ok
5,scphere,5000,0,240,28680,0.660199,0.000000e+00,ok
6,scphere,5000,1,240,28680,0.700392,0.000000e+00,ok
7,scphere,5000,2,240,28680,0.694281,0.000000e+00,ok
8,scphere,5000,3,240,28680,0.692087,0.000000e+00,ok
9,scphere,5000,4,240,28680,0.730572,0.000000e+00,ok
